# Starfield 리뷰 다국어 임베딩 (Colab)

입력: `reviews_for_embedding.csv` (분석 모집단 70,915행 · steamid + review + voted_up, `03_품질점검_분석대상확정`에서 생성)
출력: `starfield_embeddings.npz` (임베딩 + id, float16)

**실행 전 반드시:** 상단 메뉴 `런타임 > 런타임 유형 변경 > 하드웨어 가속기 = GPU (T4)`

셀을 위에서부터 순서대로 실행. 예상 소요 10~25분.

In [ ]:
# [1] GPU 확인 — GPU가 없으면 여기서 멈추고 런타임 유형부터 바꿀 것
import torch
assert torch.cuda.is_available(), "GPU가 꺼져 있음. 런타임 > 런타임 유형 변경 > GPU 선택 후 재실행"
print("GPU :", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

In [ ]:
# [2] 설치 (1~2분)
!pip install -q -U sentence-transformers
import sentence_transformers
print("sentence-transformers:", sentence_transformers.__version__)

In [ ]:
# [3] 구글 드라이브 연결
#     reviews_for_embedding.csv 를 내 드라이브 최상위에 미리 업로드해 둘 것
from google.colab import drive
drive.mount('/content/drive')

IN_PATH  = '/content/drive/MyDrive/reviews_for_embedding.csv'
OUT_PATH = '/content/drive/MyDrive/starfield_embeddings.npz'

import os
assert os.path.exists(IN_PATH), f"입력 파일 없음: {IN_PATH} — 드라이브 최상위에 업로드했는지 확인"
print("입력 파일 %.1f MB" % (os.path.getsize(IN_PATH)/1024/1024))

In [ ]:
# [4] 데이터 로드 및 검증
import pandas as pd, numpy as np

df = pd.read_csv(IN_PATH)
df['review'] = df['review'].astype(str)

print("행수          :", len(df))
print("id 고유       :", df.steamid.nunique())
print("빈 텍스트     :", int((df.review.str.len()==0).sum()))
assert len(df) == df.steamid.nunique(), "id 중복 있음"
assert (df.review.str.len() > 0).all(), "빈 텍스트 있음"

ids   = df.steamid.values
texts = []
for r in df["review"]:
    texts.append("query: " + r)      # e5 계열은 이 접두사가 필수
print("샘플:", texts[0][:80])

In [ ]:
# [5] 모델 로드 (첫 실행 시 2.24GB 다운로드, 2~3분)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')
model.max_seq_length = 512
model = model.half()          # fp16 — 속도 약 2배, 품질 차이 무시 가능

print("차원      :", model.get_sentence_embedding_dimension())   # 1024
print("최대 토큰 :", model.max_seq_length)

In [ ]:
# [6] 임베딩 — 여기가 본 작업 (10~25분)
#     OutOfMemory 가 나면 batch_size 를 64 로 낮출 것
import time
t0 = time.time()

X = model.encode(
    texts,
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print("\n소요 %.1f분" % ((time.time()-t0)/60))
print("shape:", X.shape, "| dtype:", X.dtype)

In [ ]:
# [7] 품질 검증 — 다국어 임베딩이 실제로 작동하는지 확인
#     같은 뜻(최적화 불만)이 언어가 달라도 가까운지, 반대 뜻과는 먼지
NEG = ["Give me optimization",
       "屎一样的优化，落后时代的黑屏加载",
       "컴터 2070rtx 램 32인가 64인데도 사운드 렉걸림",
       "Оптимизации нет.",
       "わかってる 神ゲーだ 頼む 最適化してくれ",
       "Vaya basura de optimizacion"]
POS = ["This game is amazing, so much fun to explore",
       "b社rpg，味道很足，量大管饱，好玩的",
       "정말 재밌어요 꼭 해보세요!!",
       "Отличная игра всем советую",
       "サブクエストの作りこみがすごいし楽しい",
       "Gran juego, muy divertido."]

probe = []
for t in NEG + POS:
    probe.append("query: " + t)
P = model.encode(probe, normalize_embeddings=True).astype(np.float32)
S = P @ P.T          # 코사인 유사도 행렬 (정규화된 벡터끼리의 내적)

# 불만끼리, 칭찬끼리, 불만↔칭찬의 평균 유사도
neg_neg = []
pos_pos = []
neg_pos = []
for i in range(6):
    for j in range(6):
        if i < j:
            neg_neg.append(S[i, j])
            pos_pos.append(S[i + 6, j + 6])
        neg_pos.append(S[i, j + 6])
nn = float(np.mean(neg_neg))
pp = float(np.mean(pos_pos))
np_ = float(np.mean(neg_pos))
print("불만↔불만 (다른 언어끼리): %.3f" % nn)
print("칭찬↔칭찬 (다른 언어끼리): %.3f" % pp)
print("불만↔칭찬 (뜻이 반대)    : %.3f" % np_)
print()
if min(nn, pp) > np_:
    print("판정: 통과 — 언어가 아니라 의미로 정렬됨")
else:
    print("판정: 실패 — 접두사/모델 확인 필요")

# 본 데이터 이상치 확인
print()
print("NaN 포함 :", bool(np.isnan(X).any()))
print("노름 평균: %.4f (1.0에 가까워야 정상)" % np.linalg.norm(X.astype(np.float32), axis=1).mean())


In [ ]:
# [8] 저장 — 반드시 id를 함께 저장해야 로컬에서 정확히 병합됨
X16 = X.astype(np.float16)          # 용량 절반, 후속 분석에 영향 없음
np.savez_compressed(OUT_PATH, embeddings=X16, steamid=ids)

import os
print("저장 완료:", OUT_PATH)
print("파일 크기: %.0f MB" % (os.path.getsize(OUT_PATH)/1024/1024))

# 즉시 재로드 검증
z = np.load(OUT_PATH)
print("재로드 shape:", z['embeddings'].shape, "| id 수:", len(z['steamid']))
assert z['embeddings'].shape[0] == len(z['steamid']) == len(df)
print("검증 통과 — 드라이브에서 다운로드하면 됨")